# Final Project Kickoff

This workbook is for today's private PythonAcademy session. Run it from top to bottom, type directly into the marked activity cells, and save your copy.

Today has two jobs: establish a working Python-and-pandas notebook baseline in the Windows VDI, and turn the course topics into the first draft of a final-project proposal. Docker and Ollama are inventory questions today, not installation tasks.

The repository map below helps newcomers find the earlier material. It is a catch-up map, not a substitute for completing those modules before project development begins.

## Windows VDI setup

### 1. Check for Python

Open PowerShell in the VDI and try the first command. If it is not recognized, try the Windows launcher alternative.

```powershell
python --version
py -3 --version
```

If neither command works, record that as an IT blocker. Installing Python itself is outside today's session. Follow the instructor's shared execution so you can still complete the workbook on paper or in the activity cells later.

### 2. Create a session folder and virtual environment

Run one of the two virtual-environment commands, using the same Python command that worked above.

```powershell
New-Item -ItemType Directory -Force -Path .\PythonAcademy\Session1
Set-Location .\PythonAcademy\Session1
python -m venv .venv
```

Windows launcher alternative:

```powershell
py -3 -m venv .venv
```

### 3. Install the notebook baseline without activating the environment

PowerShell execution policy can block activation scripts. We do not need to activate the environment. Call its Python executable directly:

```powershell
.\.venv\Scripts\python.exe -m pip install pandas ipykernel
.\.venv\Scripts\python.exe -m ipykernel install --user --name pythonacademy-vdi --display-name "PythonAcademy VDI"
```

If package installation is blocked, copy the exact failing command and the short error message into your notes. Do not paste credentials, tokens, internal URLs, or personal data. Then switch to read-along mode and follow the instructor's shared execution.

### 4. Open the notebook and select the kernel

```powershell
code .
```

In VS Code, open the student notebook. Use **Select Kernel** in the upper-right corner and choose **PythonAcademy VDI**. Run the environment check below.

### Setup status

Keep track of the highest step you completed:

1. Python command found.
2. `.venv` created.
3. `pandas` and `ipykernel` installed.
4. `PythonAcademy VDI` selected and the environment check runs.

If you stop before step 4, save the exact command that failed and a short, sanitized error. Continue in read-along mode.

In [ ]:
from pathlib import Path
import platform
import sys

import pandas as pd

python_version = platform.python_version()
pandas_version = pd.__version__
kernel_executable = Path(sys.executable).name
in_virtual_environment = sys.prefix != sys.base_prefix
ready = bool(python_version and pandas_version and kernel_executable)

print(f"Python: {python_version}")
print(f"pandas: {pandas_version}")
print(f"Operating system: {platform.system()}")
print(f"Kernel executable: {kernel_executable}")
print(f"Virtual environment detected: {in_virtual_environment}")
print(f"READY: {ready}")

### Teams-chat checkpoint 1: environment

Paste one line in Teams chat:

- `ENV: ready | step 4 | Python <version> | pandas <version>`
- or `ENV: blocked | step <number> | command: <sanitized command> | error: <short error>`

Never paste a password, token, credential, personal file path, or internal URL.

### Tool inventory only

Do not install or configure Docker or Ollama today. If time allows, these commands tell you whether they are already available:

```powershell
docker --version
ollama --version
```

Record `available`, `not found`, or `not checked` for each. A missing tool does not block today's notebook work.

## Runnable foundations recap

### Public repository catch-up map

Use the public PythonAcademy repository only as a reading and practice map:

- `CorePython/`: types, collections, control flow, functions, files, CSV, JSONL, exceptions, OOP, validation, and unit testing.
- `Numpy_Pandas/`: array and DataFrame inspection, cleaning, type conversion, transformation, joining, and reporting.
- `IntermediatePython/`: FastAPI and Pydantic, local LLM work, Docker, and workflow automation.
- `LabSession/`: integrated exercises and data-pipeline practice.

Newcomers need the earlier material before the development month. For today, we will run one small path through functions, pandas, explicit validation, and assertions.

In [ ]:
workflow_events = [
    {"event_id": "EV-001", "status": " Complete ", "duration_minutes": "12", "source": "file"},
    {"event_id": "EV-002", "status": "FAILED", "duration_minutes": "8", "source": "api"},
    {"event_id": "EV-003", "status": "pending", "duration_minutes": "", "source": "file"},
    {"event_id": "EV-004", "status": "unknown", "duration_minutes": "-3", "source": "web"},
    {"event_id": "EV-005", "status": "in progress", "duration_minutes": "19", "source": "database"},
]

events = pd.DataFrame(workflow_events)
events

In [ ]:
print(f"Shape: {events.shape}")
print("\nColumn types before cleaning:")
print(events.dtypes)
print("\nMissing values before cleaning:")
print(events.replace("", pd.NA).isna().sum())

Before running the next cells, find two values that need cleaning or validation. Write them in your own notes. We are looking for evidence in the data, not guessing what a client meant.

In [ ]:
def normalize_status(value: str) -> str:
    """Return a status in the format used by the application."""
    return value.strip().lower().replace(" ", "_")


for raw_status in events["status"]:
    print(f"{raw_status!r} -> {normalize_status(raw_status)!r}")

In [ ]:
# TODO: Change this value and predict the result before you run the cell.
status_to_try = " Pending "
normalized_try = normalize_status(status_to_try)
print(normalized_try)
assert normalized_try == "pending"

In [ ]:
clean_events = events.copy()
clean_events["status"] = clean_events["status"].map(normalize_status)
clean_events["duration_minutes"] = pd.to_numeric(
    clean_events["duration_minutes"], errors="coerce"
)

ALLOWED_STATUSES = {"pending", "in_progress", "complete", "failed"}
clean_events["status_is_valid"] = clean_events["status"].isin(ALLOWED_STATUSES)
clean_events["duration_is_valid"] = (
    clean_events["duration_minutes"].notna()
    & clean_events["duration_minutes"].ge(0)
)
clean_events["row_is_valid"] = (
    clean_events["status_is_valid"] & clean_events["duration_is_valid"]
)

clean_events

In [ ]:
valid_events = clean_events.loc[clean_events["row_is_valid"]].copy()
invalid_events = clean_events.loc[~clean_events["row_is_valid"]].copy()

print("Valid rows:")
print(valid_events[["event_id", "status", "duration_minutes"]].to_string(index=False))
print("\nInvalid rows to review:")
print(
    invalid_events[
        ["event_id", "status", "duration_minutes", "status_is_valid", "duration_is_valid"]
    ].to_string(index=False)
)

In [ ]:
def summarize_events(frame: pd.DataFrame) -> dict:
    """Summarize rows that have already passed validation."""
    return {
        "valid_event_count": len(frame),
        "complete_count": int(frame["status"].eq("complete").sum()),
        "failed_count": int(frame["status"].eq("failed").sum()),
        "average_duration_minutes": round(frame["duration_minutes"].mean(), 1),
    }


recap_summary = summarize_events(valid_events)
recap_summary

In [ ]:
assert normalize_status(" In Progress ") == "in_progress"
assert set(valid_events["status"]).issubset(ALLOWED_STATUSES)
assert valid_events["duration_minutes"].notna().all()
assert valid_events["duration_minutes"].ge(0).all()
assert len(valid_events) == 3
assert len(invalid_events) == 2
assert recap_summary["complete_count"] == 1

print("Assertions passed: cleaned rows and summary match the stated rules.")

### Teams-chat checkpoint 2: recap validation

Paste one line in Teams chat:

`RECAP: valid=3 | invalid=2 | assertions=passed`

If your result differs, paste `RECAP: check needed` and name the first cell that produced a different result.

## Course architecture recap

A project connects the earlier topics as one system:

```text
client problem
    -> source data
    -> ingestion
    -> validation and transformation
    -> business logic
    -> FastAPI
    -> automation or local-LLM component
    -> Dockerized delivery
    -> tests, documentation, and client demonstration
```

- Python types, collections, control flow, functions, files, CSV, JSONL, exceptions, and OOP structure the application.
- NumPy and pandas support inspection, cleaning, transformation, joining, and reporting.
- Explicit validation and pytest protect expected behavior.
- FastAPI and Pydantic define service and data contracts.
- Docker makes execution reproducible.
- Ollama and prompting can add local-LLM behavior.
- Requests, Beautiful Soup, Selenium, and workflow automation can handle ingestion or repeated actions.

Each arrow is a boundary. At a boundary, ask what goes in, what must be true, what comes out, how it can fail, and what evidence would show that it worked.

### Activity: trace one system boundary

Work individually. You may trace a possible client workflow or use today's synthetic `workflow_events` path for practice. Do not invent confidential details. Fill as many fields as you can; an empty field is a useful sign that you need evidence.

In [ ]:
# TODO: Replace the empty strings with your current understanding.
# Use role names and sanitized data descriptions, never real client secrets.
trace_system = {
    "client_problem": "",
    "source_data": "",
    "ingestion": "",
    "validation_and_transformation": "",
    "business_logic": "",
    "api_behavior": "",
    "specialization": "",  # web automation or local LLM
    "failure": "",
    "evidence": "",
}

missing_trace_fields = [name for name, value in trace_system.items() if not value.strip()]
print("Fields still needing evidence:", missing_trace_fields)

### Teams-chat checkpoint 3: architecture boundary

Choose one arrow in the architecture and paste one sentence:

`TRACE: At <boundary>, <input> becomes <output>. It can fail when <failure>. We would check <evidence>.`

Examples of boundaries include ingestion to validation, transformation to business logic, and API request to response. Name the behavior; do not name a client or project that you do not actually have.

## Final-project requirements

Projects are completed by teams of two or three and must address a defensible client problem. Team matching begins in Session 2 after the individual proposal seeds can be compared.

Every project must include:

- [ ] A structured Python application, not only a notebook.
- [ ] A data-ingestion workflow using a suitable file, API, database, or web source.
- [ ] Pandas-based cleaning or transformation with explicit validation.
- [ ] A FastAPI service with Pydantic request and response contracts.
- [ ] Automated pytest coverage for business logic and API behavior.
- [ ] A Dockerfile that runs the application reproducibly.
- [ ] At least one meaningful web-automation or local-LLM component.
- [ ] Error handling, logging, configuration through environment variables, and an `.env.example` without secrets.
- [ ] A README with setup, architecture, usage, tests, limitations, and demo instructions.
- [ ] Sanitized, anonymized, synthetic, or public data only. No client secrets, credentials, personal data, or proprietary production exports.
- [ ] A working final demonstration tied to the original client outcome.

### Required deliverables

1. Project proposal and scoped problem statement.
2. Team charter and responsibility split.
3. Architecture diagram.
4. Data and API contracts.
5. Source repository with issues and reviewable team contributions.
6. Dockerized working application.
7. Automated test evidence.
8. Sanitized sample data or reproducible data-generation instructions.
9. Final README and demonstration script.
10. Short retrospective covering tradeoffs, limitations, client value, and next steps.

### Session sequence and delivery milestones

1. **Session 1:** VDI setup, foundations recap, architecture trace, project rules, and individual proposal seed.
2. **Session 2:** proposal matching, team formation, team charter, requirements, user stories, scope, architecture, data contract, and API contract.
3. **Session 3:** build a vertical slice from ingestion through transformation, API, and tests.
4. **Session 4:** add the automation or LLM component, Docker packaging, reliability, and security review.
5. **Session 5:** prototype review, backlog refinement, team responsibilities, and launch plan for the development month.

After Session 5, teams have a minimum four-week development period:

- **Week 1:** approved proposal, architecture, repository structure, issues, and team roles.
- **Week 2:** working vertical slice and initial tests.
- **Week 3:** completed integration, error handling, and Docker execution.
- **Week 4:** hardening, documentation, client-value evidence, and demo rehearsal.
- **Final event:** formal team demonstration and technical review.

Exact dates for Sessions 2-5, the development month, and the final event will be scheduled separately.

### 100-point rubric

| Area | Points | Evidence to make visible |
|---|---:|---|
| Client problem, user outcome, and scope | 15 | A defensible problem, clear user outcome, and controlled scope |
| Working full-stack integration | 25 | Ingestion, data work, API, and specialization work together |
| Data quality, validation, error handling, and tests | 20 | Rules and failures are explicit and automated tests provide evidence |
| Architecture and code organization | 15 | Responsibilities and boundaries are clear in the structure and diagram |
| Docker reproducibility and documentation | 15 | Another reviewer can set up, run, test, and understand the application |
| Team process and final demonstration | 10 | Contributions are reviewable and the demo proves the client outcome |
| **Total** | **100** | |

### Requirements check

Before drafting a proposal, verify that you can answer these questions:

- What makes the problem defensible from the user's point of view?
- Where will safe data come from?
- What primary request-to-response behavior could the API expose?
- Which workflow step could justify web automation or a local LLM?
- What evidence would connect the final demonstration to the user's outcome?

## Individual proposal seed

This is not a final proposal and it does not assign you to a team. Describe what you currently know, keep client information sanitized, and make uncertainty visible. A problem statement should start with the user and current workflow, not with a technology.

In [ ]:
# TODO: Replace the empty strings with a concise, sanitized proposal seed.
proposal_seed = {
    "user": "",
    "current_workflow": "",
    "problem": "",
    "desired_outcome": "",
    "safe_data": "",
    "provisional_api_behavior": "",
    "possible_specialization": "",  # web automation or local LLM
    "uncertainty": "",
    "next_action": "",
}

missing_proposal_fields = [
    name for name, value in proposal_seed.items() if not value.strip()
]
print("Fields still needing work:", missing_proposal_fields)

if all(proposal_seed[name].strip() for name in ["user", "problem", "desired_outcome"]):
    proposal_statement = (
        f"I am exploring how to help {proposal_seed['user']} "
        f"address {proposal_seed['problem']} so that {proposal_seed['desired_outcome']}."
    )
    print(proposal_statement)
else:
    print("Complete user, problem, and desired_outcome to generate the proposal statement.")

### Proposal quality check

Your seed is ready for comparison when it includes:

- a user role and the current workflow;
- an observable problem and desired outcome;
- safe, simulated, or public data;
- one provisional API behavior;
- a possible job for web automation or a local LLM;
- one uncertainty and one concrete next action.

Do not invent details just to fill a field. State what you need to investigate.

### Session 2 preview: team charter

Before Session 2, proposal seeds will be compared for compatibility around the user, workflow, data, or desired outcome. Teams of two or three will then record members, client or user, current problem, expected outcome, safe data, primary API behavior, specialization, initial responsibilities, and the first owned task.

You do not complete the team charter today. Keep your individual proposal seed available for the matching process.

## Teams-chat checkpoint 4: exit ticket

Paste two short lines in Teams chat:

`PROPOSAL: I am exploring how to help <user> address <problem> so that <desired outcome>.`

`UNCERTAINTY: I still need to investigate <specific uncertainty>. NEXT: <concrete action>.`

Save your notebook after posting. Do not include confidential client names, personal data, credentials, or proprietary details.